# 05 — Fit / Shortlisting Predictor (Model B, Supervised — Optional)
SmartHire Phase 3.

Input: engineered features from a (resume, job) pair → Output: fit probability.

**Prerequisites:** Phase 1 + 2 artifacts must exist (`python -m src.models.recommender` and `python -m src.models.clustering` must have been run).

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import RocCurveDisplay, roc_curve, auc

from src.config import FIGURES_DIR
from src.models.fit_predictor import (
    load_clean_resumes, train_and_compare, save_fit_predictor
)
from src.models.recommender import load_job_corpus, load_job_vectors
from src.features.match_features import build_training_features
from src.config import RANDOM_STATE, NEG_POS_RATIO

sns.set_style('whitegrid')

## 1. Load Phase 1 + 2 artifacts

In [ ]:
resume_df  = load_clean_resumes()
job_corpus = load_job_corpus()
job_vectors, job_vectorizer = load_job_vectors()
print(f'Resumes: {len(resume_df)} | Jobs: {len(job_corpus)} | Vectors: {job_vectors.shape}')

## 2. Build synthetic (resume, job) training pairs

Positive = resume paired with a job whose title matches the resume category.
Negative = resume paired with a random mismatched job (ratio 1:3).

In [ ]:
X, y, feature_names = build_training_features(
    resume_df, job_corpus, job_vectors, job_vectorizer,
    neg_pos_ratio=NEG_POS_RATIO, random_state=RANDOM_STATE,
)
print(f'Training set: {len(y)} pairs | features={feature_names}')
print(f'Class balance: positives={y.sum()} ({y.mean():.1%}), negatives={len(y)-y.sum()}')

## 3. Train and compare models

In [ ]:
best_name, best_model, scaler, results = train_and_compare(X, y, feature_names)

results_df = pd.DataFrame(results).T.sort_values('roc_auc', ascending=False)
results_df

## 4. ROC curve comparison

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

fig, ax = plt.subplots(figsize=(7, 5))

for name, (model_cls, needs_scale) in [
    ('Logistic Regression', (LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE), True)),
    ('XGBoost',             (XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                                           use_label_encoder=False, eval_metric='logloss',
                                           random_state=RANDOM_STATE, verbosity=0), False)),
]:
    Xtr, Xte = (StandardScaler().fit(X_train).transform(X_train),
                StandardScaler().fit(X_train).transform(X_test)) if needs_scale else (X_train, X_test)
    model_cls[0].fit(Xtr, y_train)
    proba = model_cls[0].predict_proba(Xte)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f'{name} (AUC={roc_auc:.3f})')

ax.plot([0,1],[0,1], 'k--', label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — Fit Predictor')
ax.legend()
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'fit_predictor_roc_curve.png', dpi=150)
plt.show()

## 5. Feature importance

In [ ]:
from xgboost import XGBClassifier
xgb = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                    use_label_encoder=False, eval_metric='logloss',
                    random_state=RANDOM_STATE, verbosity=0)
xgb.fit(X_train, y_train)

imp_df = pd.DataFrame({'feature': feature_names, 'importance': xgb.feature_importances_})
imp_df = imp_df.sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(6, 4))
ax.barh(imp_df['feature'], imp_df['importance'], color='steelblue')
ax.set_title('XGBoost Feature Importance — Fit Predictor')
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'fit_predictor_feature_importance.png', dpi=150)
plt.show()

## 6. Save the best model

In [ ]:
save_fit_predictor(best_model, scaler, feature_names)

## 7. Notes / findings
- Best model and ROC-AUC: ...
- Most predictive feature (from XGB importance): ...
- Limitations of the synthetic training approach: ...
- How to improve with real labelled shortlisting data if available: ...